# Notebook Semana 1: Primeros pasos con angr

**Objetivo:** familiarizarte con angr hasta el punto de poder cargar un binario ELF x86-64, recuperar su CFG, iterar por funciones y bloques básicos, y acceder a las instrucciones desensambladas.

Al terminar este notebook deberías ser capaz de:

1. Cargar un binario con `angr.Project` y entender qué te devuelve.
2. Lanzar `CFGFast` y entender la estructura del grafo que produce.
3. Navegar funciones → bloques básicos → instrucciones.
4. Identificar llamadas a funciones externas (libc) mirando la PLT.
5. Detectar syscalls directas.
6. Serializar una primera versión muy cruda del CFG a JSON.

Esto es código de **exploración**. No es el código de producción del prototipo. Pero lo que aprendas aquí es la base sobre la que construirás el Extractor en la semana 3.

---

## 0. Preparación del entorno

Antes de ejecutar nada, instala las dependencias en un entorno virtual:

```bash
python3 -m venv venv
source venv/bin/activate
pip install angr capstone pyelftools ipykernel
```

Si angr da problemas al instalar (suele dar warnings de dependencias de bajo nivel), en Debian/Ubuntu necesitas:

```bash
sudo apt install build-essential python3-dev libffi-dev libssl-dev
```

Esto a veces tarda 5-10 minutos. Es normal.

## 1. Preparar un binario de prueba

Crea un programa C sencillo para experimentar. Desde terminal:

```bash
mkdir -p binarios_prueba
cat > binarios_prueba/hello.c <<'EOF'
#include <stdio.h>

int suma(int a, int b) {
    return a + b;
}

int main(void) {
    int x = suma(3, 4);
    printf("resultado: %d\n", x);
    return 0;
}
EOF

gcc -O0 -no-pie binarios_prueba/hello.c -o binarios_prueba/hello
```

Dos notas importantes:

- **`-O0`**: sin optimización. GCC no reordena ni inline-a nada. Para empezar esto es lo que quieres.
- **`-no-pie`**: sin Position Independent Executable. Las direcciones son fijas y fáciles de razonar. Los binarios modernos suelen ser PIE; con PIE activo las direcciones que ves en angr son relativas a una base aleatoria. Para aprender es más fácil sin PIE.

Verifica con `file binarios_prueba/hello` que sale algo como `ELF 64-bit LSB executable, x86-64`.

In [ ]:
# Ajusta esta ruta a donde tengas el binario
BINARIO = 'binarios_prueba/hello'

import os
assert os.path.exists(BINARIO), f'No encuentro el binario en {BINARIO}. Compila primero el hello.c'
print(f'Binario a analizar: {BINARIO} ({os.path.getsize(BINARIO)} bytes)')

## 2. Cargar el binario con angr

`angr.Project` es el punto de entrada. Cuando cargas un binario, angr:

- Parsea cabeceras ELF (usa pyelftools por debajo).
- Carga las librerías dinámicas necesarias (libc, ld).
- Prepara un modelo de memoria.
- Identifica el punto de entrada.

El parámetro `auto_load_libs=False` es **muy importante**: sin él angr intenta cargar libc completa, lo que puede tardar mucho y no lo necesitas para construir el CFG de tu programa.

In [ ]:
import angr
import logging

# angr es muy verboso por defecto. Silenciamos los warnings para que el notebook sea legible.
logging.getLogger('angr').setLevel(logging.ERROR)
logging.getLogger('cle').setLevel(logging.ERROR)
logging.getLogger('pyvex').setLevel(logging.ERROR)

proj = angr.Project(BINARIO, auto_load_libs=False)

print(f'Arquitectura: {proj.arch.name}')
print(f'Bits: {proj.arch.bits}')
print(f'Endianness: {proj.arch.memory_endness}')
print(f'Entry point: {hex(proj.entry)}')
print(f'Filename: {proj.filename}')
print(f'Main object: {proj.loader.main_object}')

### Inspección del objeto cargado

`proj.loader` te da acceso al ELF parseado. Es equivalente a lo que pyelftools te daría por separado, pero ya viene resuelto con otras librerías cargadas.

In [ ]:
main_obj = proj.loader.main_object

print('Secciones:')
for section in main_obj.sections:
    if section.name:
        print(f'  {section.name:20s} en {hex(section.vaddr):12s} tamaño {section.memsize} bytes')

print('\nSímbolos importados (primeros 10):')
for sym in list(main_obj.symbols)[:10]:
    if sym.is_import:
        print(f'  {sym.name}')

print('\nSímbolos exportados/definidos (primeros 10):')
for sym in list(main_obj.symbols)[:20]:
    if sym.is_function and sym.is_export:
        print(f'  {sym.name:30s} en {hex(sym.rebased_addr)}')

## 3. Construir el CFG con CFGFast

angr tiene dos recuperadores principales de CFG:

- **`CFGFast`**: análisis estático, rápido (segundos), cobertura buena en binarios no ofuscados. Lo que vas a usar por defecto.
- **`CFGEmulated`**: usa ejecución simbólica. Más lento (minutos a horas), más preciso con saltos indirectos. Para el TFE, empieza con CFGFast y solo considera CFGEmulated para casos concretos.

**Atención:** el parámetro `normalize=True` es importante. Sin él, un mismo bloque puede aparecer partido en dos si hay saltos que caen en medio. Con él, angr garantiza que cada bloque básico tiene una sola entrada y una sola salida. Es lo que necesitas para traducir a C de forma razonable.

In [ ]:
cfg = proj.analyses.CFGFast(normalize=True)

print(f'Funciones detectadas: {len(cfg.functions)}')
print(f'Nodos del CFG: {len(cfg.graph.nodes())}')
print(f'Aristas del CFG: {len(cfg.graph.edges())}')

### Listar las funciones del programa

angr detecta tanto las funciones que tú escribiste (main, suma) como las que forman parte del runtime de GCC (`_start`, `__libc_start_main@plt`, etc.). Para el TFE normalmente te interesan las tuyas.

In [ ]:
print(f'{"Dirección":<12} {"Nombre":<40} {"Bloques":<8} {"PLT?":<6}')
print('-' * 70)

for func_addr, func in cfg.functions.items():
    es_plt = '  *' if func.is_plt else ''
    print(f'{hex(func_addr):<12} {func.name:<40} {len(func.blocks):<8} {es_plt}')

Identifica las funciones que son tuyas (`main`, `suma`). Verás que además aparecen:

- `_start`: punto de entrada real del programa, lo pone el enlazador. Llama a `__libc_start_main` que a su vez llama a tu `main`.
- Funciones `__libc_start_main`, `printf`, etc. marcadas como `is_plt=True`: son trampolines de la PLT que resuelven llamadas dinámicas a libc.
- Funciones con nombres raros como `frame_dummy`, `__do_global_dtors_aux`: glue del runtime de C. Puedes ignorarlas en tu traductor.

### Inspeccionar una función concreta

In [ ]:
# Buscar la función 'suma' que escribimos nosotros
func_suma = cfg.functions.function(name='suma')
assert func_suma is not None, 'No encontré la función suma. ¿Compilaste el binario correcto?'

print(f'Función: {func_suma.name}')
print(f'Dirección: {hex(func_suma.addr)}')
print(f'Tamaño: {func_suma.size} bytes')
print(f'Número de bloques básicos: {len(func_suma.blocks)}')
print(f'Es PLT: {func_suma.is_plt}')
print(f'Llama a: {[called.name for called in func_suma.functions_called() if called.name]}')
print(f'Es llamada desde: {[caller.name for caller in func_suma.functions_calling() if caller.name]}')

### Iterar por los bloques básicos de una función

In [ ]:
print(f'Bloques básicos de {func_suma.name}:\n')

for block in func_suma.blocks:
    print(f'  Bloque en {hex(block.addr)} ({block.size} bytes, {block.instructions} instrucciones)')
    # Ver las instrucciones desensambladas
    for insn in block.capstone.insns:
        print(f'    {hex(insn.address):12s}  {insn.mnemonic:8s} {insn.op_str}')
    print()

**Importante:** fíjate en el atributo `block.capstone`. Eso es angr dándote directamente capstone para el contenido del bloque. Cada `insn` es un objeto `capstone.CsInsn` con toda la riqueza de capstone: operandos desmenuzados, grupos de instrucción, registros leídos y escritos, etc.

Experimenta:

In [ ]:
# Tomar la primera instrucción de suma y ver qué nos da capstone
first_block = next(iter(func_suma.blocks))
first_insn = first_block.capstone.insns[0]

print(f'Instrucción: {first_insn.mnemonic} {first_insn.op_str}')
print(f'Dirección: {hex(first_insn.address)}')
print(f'Bytes: {first_insn.bytes.hex()}')
print(f'Tamaño: {first_insn.size}')
print(f'Registros leídos: {[first_insn.reg_name(r) for r in first_insn.regs_access()[0]]}')
print(f'Registros escritos: {[first_insn.reg_name(r) for r in first_insn.regs_access()[1]]}')
print(f'Grupos: {[first_insn.group_name(g) for g in first_insn.groups]}')

## 4. Aristas del CFG: cómo se conectan los bloques

El CFG tiene nodos (bloques) y aristas (transferencias de control). Cada arista tiene un tipo que te dice si es caída natural, salto condicional, llamada, retorno, etc. Esto es **crítico** para tu traductor.

angr expone las aristas en `cfg.graph` (un `networkx.DiGraph`) y en el atributo `transition_graph` de cada función.

In [ ]:
func_main = cfg.functions.function(name='main')

print(f'Transiciones dentro de main:\n')
for src, dst, data in func_main.transition_graph.edges(data=True):
    tipo = data.get('type', 'fall_through')
    src_addr = getattr(src, 'addr', '?')
    dst_addr = getattr(dst, 'addr', '?')
    src_str = hex(src_addr) if isinstance(src_addr, int) else str(src)
    dst_str = hex(dst_addr) if isinstance(dst_addr, int) else str(dst)
    print(f'  {src_str} -> {dst_str}  ({tipo})')

Los tipos de arista más comunes que vas a ver son:

- `fall_through`: caída natural a la siguiente instrucción (por ejemplo, al final de un bloque sin salto).
- `transition`: salto condicional o incondicional interno a la función.
- `call`: llamada a otra función.
- `return`: retorno.
- `fake_ret`: arista artificial que angr añade para que tras un `call` se pueda seguir analizando la caída.

Esta información es **exactamente** lo que vas a necesitar para emitir los `goto` y los `if` del C generado.

## 5. Detectar llamadas a funciones externas

Esto es la primera anotación semántica del CFG enriquecido. Cuando veas una llamada a una función que está en la PLT, sabes que es una llamada dinámica a una librería externa (típicamente libc).

Dos formas complementarias de detectarlo:

1. La función destino tiene `is_plt == True`.
2. El nombre de la función coincide con un símbolo importado.

In [ ]:
print('Llamadas externas detectadas en el binario:\n')

for func_addr, func in cfg.functions.items():
    if func.is_plt:
        continue  # saltamos las PLT stubs en sí mismas
    if func.name in ['_start', 'frame_dummy', '_init', '_fini', '__libc_csu_init',
                      '__libc_csu_fini', 'register_tm_clones', 'deregister_tm_clones',
                      '__do_global_dtors_aux']:
        continue  # saltamos glue de C runtime
    
    llamadas_externas = []
    for called in func.functions_called():
        if called.is_plt:
            llamadas_externas.append(called.name)
    
    if llamadas_externas:
        print(f'  Función {func.name} llama a: {", ".join(llamadas_externas)}')

## 6. Detectar syscalls directas

En programas escritos en C normalmente las syscalls se hacen a través de libc (`write`, `read`, etc.) y lo que ves son llamadas a esas funciones de libc. Pero en binarios más crudos (o código assembly inline) puedes encontrar instrucciones `syscall` directas. Te conviene saber detectarlas aunque hello world no tenga ninguna.

Busca instrucciones cuyo mnemónico sea `syscall`. El número de syscall está típicamente en el registro `rax` justo antes, cargado con una instrucción `mov`.

In [ ]:
print('Buscando instrucciones syscall en todo el binario...\n')

encontradas = 0
for func_addr, func in cfg.functions.items():
    for block in func.blocks:
        for insn in block.capstone.insns:
            if insn.mnemonic == 'syscall':
                print(f'  syscall en {hex(insn.address)} (función {func.name})')
                encontradas += 1

if encontradas == 0:
    print('  (ninguna syscall directa; es lo esperado en un binario C compilado con glibc)')

Para probarlo con un binario que sí tenga syscalls directas, puedes compilar un programa con inline assembly:

```c
#include <unistd.h>
int main(void) {
    const char *msg = "hola via syscall\n";
    long ret;
    asm volatile ("syscall" : "=a"(ret) : "0"(1), "D"(1), "S"(msg), "d"(17) : "rcx", "r11", "memory");
    return 0;
}
```

O más simple, un binario escrito en assembly puro. Déjalo como experimento opcional.

## 7. Primer volcado del CFG a JSON (versión embrionaria)

Este es el primer paso hacia el Extractor que implementarás en la semana 3. La estructura es muy cruda aún — no hay anotaciones semánticas, no hay distinción por tipo de arista, no hay metadatos — pero sirve para:

1. Confirmar que sabes navegar angr.
2. Ver el tamaño real del CFG sobre disco.
3. Empezar a pensar el esquema definitivo del JSON.

In [ ]:
import json

def cfg_a_dict_crudo(cfg, proj):
    """Volcado mínimo del CFG a un diccionario serializable.
    
    Esto NO es el esquema definitivo. Es una primera aproximación para ver qué
    forma va tomando. El esquema final se fija en la semana 3.
    """
    resultado = {
        'binario': proj.filename,
        'arquitectura': proj.arch.name,
        'entry_point': hex(proj.entry),
        'funciones': {}
    }
    
    for func_addr, func in cfg.functions.items():
        # Saltamos lo que no es código de usuario para el primer experimento
        if func.is_plt:
            continue
        if func.name.startswith('_') or func.name == 'frame_dummy':
            continue
        
        bloques = {}
        for block in func.blocks:
            instrucciones = []
            for insn in block.capstone.insns:
                instrucciones.append({
                    'addr': hex(insn.address),
                    'mnemonic': insn.mnemonic,
                    'operands': insn.op_str,
                    'size': insn.size
                })
            bloques[hex(block.addr)] = {
                'size': block.size,
                'instrucciones': instrucciones
            }
        
        # Aristas internas de la función
        aristas = []
        for src, dst, data in func.transition_graph.edges(data=True):
            if isinstance(getattr(src, 'addr', None), int) and isinstance(getattr(dst, 'addr', None), int):
                aristas.append({
                    'src': hex(src.addr),
                    'dst': hex(dst.addr),
                    'tipo': data.get('type', 'fall_through')
                })
        
        resultado['funciones'][hex(func_addr)] = {
            'nombre': func.name,
            'bloques': bloques,
            'aristas': aristas,
            'llama_a_externas': [c.name for c in func.functions_called() if c.is_plt]
        }
    
    return resultado


datos = cfg_a_dict_crudo(cfg, proj)

# Imprimimos las primeras líneas del JSON
json_str = json.dumps(datos, indent=2)
print(json_str[:2000])
print(f'\n[...]\n\nTotal: {len(json_str)} caracteres')

# Guardamos para que puedas verlo completo con un editor
with open('primer_cfg.json', 'w') as f:
    f.write(json_str)
print(f'\nGuardado en primer_cfg.json')

## 8. Ejercicios para fijar lo aprendido

Para terminar la semana 1 sólidamente, intenta resolver estos ejercicios sobre el mismo binario o sobre otro que compiles tú:

**Ejercicio 1 (fácil):** escribe una función que reciba una dirección de instrucción y te devuelva a qué función y a qué bloque básico pertenece.

**Ejercicio 2 (fácil):** modifica el volcado JSON para que cada instrucción incluya también los bytes en hexadecimal (campo `bytes` usando `insn.bytes.hex()`).

**Ejercicio 3 (medio):** compila un binario con un `switch` de 5 ramas (`switch (x) { case 1: ... case 2: ...}` ) y busca si en tu CFG aparecen saltos indirectos. Pista: el CFG de angr marca los bloques con `jumpkind`. Busca en `block.vex.jumpkind` o en las aristas con tipo relevante.

**Ejercicio 4 (medio):** compila el mismo hello con `-O0` y con `-O2` y compara los CFGs. ¿Cuántas funciones detecta angr en cada caso? ¿Qué tamaño tienen los bloques básicos? ¿Es más fácil o más difícil razonar sobre el CFG de `-O2`?

**Ejercicio 5 (difícil, opcional):** escribe el embrión de un traductor que, dado un bloque básico, emita texto C con un comentario por cada instrucción (aunque no traduzca nada aún, solo ponga `// 0x401140: mov rbp, rsp`). Si terminas esto, tienes ya en las manos el germen del Traductor que trabajarás la semana 4.

## 9. Dónde mirar cuando te atasques

- Documentación oficial de angr: https://docs.angr.io/
- Tutorial oficial (el de *Top Self-Study Course*): https://docs.angr.io/en/latest/getting-started/index.html
- API reference de `CFGFast`: https://api.angr.io/angr.analyses.html#angr.analyses.cfg.cfg_fast.CFGFast
- Documentación de capstone Python: https://www.capstone-engine.org/lang_python.html
- Canal de Discord de angr (muy activo): https://angr.io/ → enlace a Slack/Discord

**Consejo importante:** angr es una librería compleja y hay cosas que solo se entienden mirando el código. No dudes en abrir el código fuente de angr cuando algo no cuadre con la documentación. La documentación tiene lagunas pero el código está razonablemente bien estructurado.

## Checklist de fin de semana 1

Cuando termines esta semana, deberías poder tachar estos puntos:

- [ ] Entorno virtual con angr, capstone, pyelftools funcionando.
- [ ] Binario de prueba compilado y cargado correctamente en angr.
- [ ] CFG construido con `CFGFast(normalize=True)`.
- [ ] Sabes iterar por funciones, bloques e instrucciones.
- [ ] Sabes distinguir funciones PLT de funciones de usuario.
- [ ] Sabes acceder a los tipos de arista del CFG.
- [ ] Ejercicios 1 y 2 resueltos.
- [ ] Tienes un `primer_cfg.json` generado correctamente.

Si llegas al final de la semana sin tachar todos los puntos, **no pasa nada** — ajustas el inicio de la semana 2. Pero si te faltan más de dos, es señal de que necesitas acotar más el alcance del TFE antes de seguir.